# LeetCode 2295 — Replace Elements in an Array bằng Python

Link đề: https://leetcode.com/problems/replace-elements-in-an-array/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Simulation

Skeleton LeetCode:

```python
class Solution:
    def arrayChange(self, nums: List[int], operations: List[List[int]]) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Thấy vì sao mô phỏng "tìm `x` trong mảng rồi thay" là O(n·m) và cách bỏ bước tìm bằng **bảng băm giá trị → chỉ số** (chỉ mục ngược, inverted index).
2. Duy trì **bất biến** giữa hai cấu trúc dữ liệu (mảng `nums` và từ điển `pos`) sau mỗi thao tác.
3. Một góc nhìn khác: **ghép các ánh xạ theo chiều ngược** — xử lý thao tác từ cuối về đầu để biết "giá trị `v` cuối cùng biến thành gì".
4. Nhận ra cái bẫy "đi theo chuỗi `x → y → …`" khi giá trị **được dùng lại** (một giá trị bị thay đi rồi sau đó xuất hiện lại) — có thể tạo **chu trình**.
5. Mở rộng sang trường hợp mảng có **giá trị trùng** (thay mọi lần xuất hiện) — cách ghép ngược vẫn đúng nguyên vẹn.

## 2. Đề bài & ví dụ

Cho mảng `nums` gồm `n` số nguyên dương **phân biệt**. Thực hiện `m` thao tác: thao tác thứ `i` thay số `operations[i][0]` bằng `operations[i][1]`. Đảm bảo ở thao tác `i`: `operations[i][0]` **có** trong `nums` và `operations[i][1]` **không có** trong `nums` (tại thời điểm đó). Trả về mảng sau mọi thao tác.

**Ví dụ 1:** `nums = [1,2,4,6]`, `operations = [[1,3],[4,7],[6,1]]` → `[3,2,7,1]`.

**Ví dụ 2:** `nums = [1,2]`, `operations = [[1,3],[2,1],[3,2]]` → `[2,1]`.

**Ràng buộc:** `1 ≤ n, m ≤ 10⁵`; giá trị trong `[1, 10⁶]`; các giá trị của `nums` phân biệt.

In [1]:
from typing import List
import random, time

examples = [([1, 2, 4, 6], [[1, 3], [4, 7], [6, 1]], [3, 2, 7, 1]), ([1, 2], [[1, 3], [2, 1], [3, 2]], [2, 1])]
for a, ops, exp in examples:
    print(f"nums={a} ops={ops} -> kỳ vọng {exp}")

nums=[1, 2, 4, 6] ops=[[1, 3], [4, 7], [6, 1]] -> kỳ vọng [3, 2, 7, 1]
nums=[1, 2] ops=[[1, 3], [2, 1], [3, 2]] -> kỳ vọng [2, 1]


## 3. Kiến thức nền

### 3.1 `list.index` là O(n)

Tìm một giá trị trong danh sách phải quét tuần tự. `m` lần tìm ⇒ O(n·m) = `10¹⁰` worst.

In [2]:
a = list(range(10**5))
t0 = time.perf_counter()
for _ in range(200): a.index(10**5 - 1)
print(f"200 lần index phần tử cuối: {(time.perf_counter() - t0)*1000:.0f} ms (10^5 lần sẽ gấp 500 lần)")

200 lần index phần tử cuối: 140 ms (10^5 lần sẽ gấp 500 lần)


### 3.2 Chỉ mục ngược: từ điển `giá trị → chỉ số`

Vì giá trị **phân biệt**, mỗi giá trị ứng với đúng một chỉ số ⇒ `pos = {x: i for i, x in enumerate(nums)}`; tra cứu O(1) trung bình.

In [3]:
nums = [1, 2, 4, 6]
pos = {x: i for i, x in enumerate(nums)}
print(pos, "| 4 ở chỉ số", pos[4])

{1: 0, 2: 1, 4: 2, 6: 3} | 4 ở chỉ số 2


### 3.3 Hợp thành ánh xạ (function composition)

Mỗi thao tác là một ánh xạ `fᵢ(v) = y nếu v = x, ngược lại v`. Mảng cuối là `(f_m ∘ … ∘ f₁)(v)` áp cho từng phần tử ban đầu. Ghép **từ cuối về đầu** cho phép tính sẵn `F(v) = f_m(…f₁(v))` cho những `v` cần thiết mà không phải mô phỏng.

In [4]:
ops = [[1, 3], [2, 1], [3, 2]]
def apply_all(v):
    for x, y in ops:
        if v == x: v = y
    return v
print({v: apply_all(v) for v in [1, 2, 3]})

{1: 2, 2: 1, 3: 2}


### 3.4 Vì sao "không có trong nums" lại quan trọng

Đảm bảo `y` chưa có ⇒ sau thao tác, giá trị vẫn **phân biệt** ⇒ từ điển `pos` vẫn là song ánh. Nếu `y` đã có, hai chỉ số sẽ cùng giá trị và `pos[y]` mơ hồ (mục 15.2 xử lý trường hợp này).

## 4. Ý tưởng nền tảng / trực giác

Bước tốn kém duy nhất của mô phỏng là **tìm** `x`. Nếu luôn biết `x` đang ở đâu, mỗi thao tác chỉ cần:

1. `i = pos[x]` — vị trí của `x`;
2. `nums[i] = y` — thay;
3. `pos[y] = i` — giờ `y` ở vị trí `i` (có thể `del pos[x]`, nhưng không bắt buộc vì `x` không còn trong mảng; nếu sau này `x` xuất hiện lại do một thao tác khác, khóa `pos[x]` sẽ bị **ghi đè** đúng lúc đó).

Tính tay ví dụ 2:

| thao tác | `pos[x]` | nums sau | pos sau (chỉ các khóa còn ý nghĩa) |
|---|---|---|---|
| — | | [1, 2] | {1:0, 2:1} |
| 1 → 3 | 0 | [3, 2] | {3:0, 2:1} |
| 2 → 1 | 1 | [3, 1] | {3:0, 1:1} |
| 3 → 2 | 0 | [2, 1] | {2:0, 1:1} |

⇒ `[2, 1]` ✔. Chú ý: giá trị `1` bị thay đi ở thao tác đầu rồi **xuất hiện lại** ở thao tác thứ hai — đó là lý do không thể chỉ ghi "1 biến thành 3" một lần cho mãi mãi.

In [5]:
def trace_pos(nums, ops):
    nums = list(nums); pos = {x: i for i, x in enumerate(nums)}
    print("ban đầu:", nums)
    for x, y in ops:
        i = pos.pop(x); nums[i] = y; pos[y] = i
        print(f"  {x} -> {y}: chỉ số {i}, nums = {nums}, pos = {pos}")
    return nums

trace_pos(*examples[1][:2])

ban đầu: [1, 2]
  1 -> 3: chỉ số 0, nums = [3, 2], pos = {2: 1, 3: 0}
  2 -> 1: chỉ số 1, nums = [3, 1], pos = {3: 0, 1: 1}
  3 -> 2: chỉ số 0, nums = [2, 1], pos = {1: 1, 2: 0}


[2, 1]

## 5. Cách 1 — Brute Force / baseline

### 5.1 Mô phỏng bằng `list.index` — O(n · m)

In [6]:
def brute(nums, operations):
    a = list(nums)
    for x, y in operations:
        a[a.index(x)] = y
    return a

for a, ops, exp in examples:
    print(brute(a, ops), "OK" if brute(a, ops) == exp else "SAI")

[3, 2, 7, 1] OK
[2, 1] OK


Đúng theo định nghĩa nhưng worst case (luôn thay phần tử ở cuối mảng) tốn `n · m = 10¹⁰` phép so sánh.

### 5.2 Baseline 2 — Áp từng thao tác lên **mọi** phần tử (ánh xạ theo giá trị) — O(n · m)

Cách nhìn "hàm hợp" làm thẳng: với mỗi phần tử ban đầu, chạy qua cả dãy thao tác. Cũng O(n·m), nhưng là bước đệm cho cách ghép ngược.

In [7]:
def brute_compose(nums, operations):
    out = []
    for v in nums:
        for x, y in operations:
            if v == x: v = y
        out.append(v)
    return out

for a, ops, exp in examples:
    print(brute_compose(a, ops), "OK" if brute_compose(a, ops) == exp else "SAI")

[3, 2, 7, 1] OK
[2, 1] OK


**Nút thắt:** cả hai cách đều lặp "tìm kiếm" thừa. Cách 1 cần biết **vị trí** của `x` ⇒ từ điển `pos`. Cách 2 cần biết **kết quả cuối** của mỗi giá trị ⇒ tính sẵn bằng cách duyệt thao tác ngược.

## 6. Ý tưởng thuật toán / bước đột phá

### 6.1 Chỉ mục ngược (cách doocs) — O(n + m)

```
pos = {x: i for i, x in enumerate(nums)}
for x, y in operations:
    i = pos[x]; nums[i] = y; pos[y] = i
```

**Bất biến:** sau mỗi thao tác, với mọi giá trị `v` **đang có** trong `nums`, `pos[v]` là chỉ số của nó. (Các khóa của giá trị đã biến mất có thể còn sót lại, nhưng không bao giờ được tra cứu khi chúng không có trong mảng — đề đảm bảo `x` luôn có mặt.)

### 6.2 Ghép ánh xạ ngược — O(n + m)

Duyệt thao tác từ **cuối về đầu**, duy trì `final[v]` = giá trị cuối cùng mà `v` sẽ trở thành nếu `v` có mặt **ngay trước** đoạn thao tác đang xét:

- khởi đầu (sau thao tác cuối): `final[v] = v` với mọi `v` (ngầm định);
- khi lùi qua thao tác `x → y`: một giá trị `x` có mặt ngay trước thao tác này sẽ thành `y`, rồi tiếp tục thành `final[y]` ⇒ `final[x] = final.get(y, y)`. Các giá trị khác không bị thao tác này chạm tới.

Cuối cùng `answer[i] = final.get(nums[i], nums[i])`. Cách này **không cần** giá trị phân biệt — rất hữu ích cho biến thể.

### 6.3 Chọn cách nào: trực tuyến (online) hay ngoại tuyến (offline)?

| Tiêu chí | Chỉ mục ngược | Ghép ngược |
|---|---|---|
| Cần biết trước toàn bộ thao tác? | không — xử lý từng thao tác khi đến | **có** — phải duyệt từ cuối |
| Trả lời "giá trị v đang ở đâu" giữa chừng | O(1) | không hỗ trợ |
| Giá trị trùng / `y` đã tồn tại | hỏng song ánh, cần sửa nhiều | **vẫn đúng** |
| Bộ nhớ phụ | O(n) (mọi giá trị đang có) | O(số khóa xuất hiện trong thao tác) |

Với bài gốc, cả hai đều O(n + m); chỉ mục ngược là lựa chọn tự nhiên và gần với mô phỏng nhất.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Chỉ mục ngược

In [8]:
def index_map(nums, operations):
    nums = list(nums)
    d = {x: i for i, x in enumerate(nums)}
    for x, y in operations:
        nums[d[x]] = y
        d[y] = d[x]
    return nums

for a, ops, exp in examples:
    print(index_map(a, ops), "OK" if index_map(a, ops) == exp else "SAI")

[3, 2, 7, 1] OK
[2, 1] OK


### 7.2 Cách 3 — Ghép ánh xạ ngược

In [9]:
def reverse_compose(nums, operations):
    final = {}
    for x, y in reversed(operations):
        final[x] = final.get(y, y)
    return [final.get(v, v) for v in nums]

for a, ops, exp in examples:
    print(reverse_compose(a, ops), "OK" if reverse_compose(a, ops) == exp else "SAI")

[3, 2, 7, 1] OK
[2, 1] OK


### 7.3 Cách 4 — Chỉ mục ngược bằng **mảng** thay cho dict (giá trị ≤ 10⁶)

Mảng `pos` kích thước `10⁶ + 1` truy cập nhanh hơn dict, đổi lại tốn ~8 MB (list Python) — hoặc dùng `array('i')` cho gọn hơn.

In [10]:
from array import array

def index_array(nums, operations, MAXV=10**6):
    nums = list(nums)
    pos = array("i", [0]) * (MAXV + 1)
    for i, x in enumerate(nums): pos[x] = i
    for x, y in operations:
        i = pos[x]; nums[i] = y; pos[y] = i
    return nums

for a, ops, exp in examples:
    print(index_array(a, ops), "OK" if index_array(a, ops) == exp else "SAI")

[3, 2, 7, 1] OK
[2, 1] OK


### 7.4 Cách 5 — Ghép ánh xạ **xuôi** đúng cách (theo dõi "giá trị gốc nào đang mang giá trị hiện tại")

Duy trì `cur_of_origin` không tiện; thay vào đó duy trì `origin[v]` = giá trị **ban đầu** đang mang giá trị `v`. Thao tác `x → y`: `origin[y] = origin.pop(x)`. Cuối cùng đảo ánh xạ.

In [11]:
def forward_origin(nums, operations):
    origin = {v: v for v in nums}               # giá trị hiện tại -> giá trị ban đầu
    for x, y in operations:
        origin[y] = origin.pop(x)
    result_of = {o: v for v, o in origin.items()}   # giá trị ban đầu -> giá trị cuối
    return [result_of[v] for v in nums]

for a, ops, exp in examples:
    print(forward_origin(a, ops), "OK" if forward_origin(a, ops) == exp else "SAI")

[3, 2, 7, 1] OK
[2, 1] OK


## 8. TRACE từng bước

### 8.1 Trace cách 2 (chỉ mục ngược) trên ví dụ 1

In [12]:
print("kết quả:", trace_pos(*examples[0][:2]))

ban đầu: [1, 2, 4, 6]
  1 -> 3: chỉ số 0, nums = [3, 2, 4, 6], pos = {2: 1, 4: 2, 6: 3, 3: 0}
  4 -> 7: chỉ số 2, nums = [3, 2, 7, 6], pos = {2: 1, 6: 3, 3: 0, 7: 2}
  6 -> 1: chỉ số 3, nums = [3, 2, 7, 1], pos = {2: 1, 3: 0, 7: 2, 1: 3}
kết quả: [3, 2, 7, 1]


### 8.2 Trace cách 3 (ghép ngược) trên ví dụ 2

In [13]:
def trace_reverse(nums, operations):
    final = {}
    for t in range(len(operations) - 1, -1, -1):
        x, y = operations[t]
        final[x] = final.get(y, y)
        print(f"lùi qua thao tác {t} ({x} -> {y}): final[{x}] = final.get({y}, {y}) = {final[x]}   final = {final}")
    ans = [final.get(v, v) for v in nums]
    print("áp vào nums", nums, "->", ans)
    return ans

trace_reverse(*examples[1][:2])

lùi qua thao tác 2 (3 -> 2): final[3] = final.get(2, 2) = 2   final = {3: 2}
lùi qua thao tác 1 (2 -> 1): final[2] = final.get(1, 1) = 1   final = {3: 2, 2: 1}
lùi qua thao tác 0 (1 -> 3): final[1] = final.get(3, 3) = 2   final = {3: 2, 2: 1, 1: 2}
áp vào nums [1, 2] -> [2, 1]


[2, 1]

Đọc: lùi qua `3 → 2` (thao tác cuối): giá trị 3 có mặt trước thao tác này sẽ kết thúc là 2. Lùi qua `2 → 1`: 2 thành 1, rồi 1 không bị thao tác nào sau đó đổi ⇒ `final[2] = 1`. Lùi qua `1 → 3`: 1 thành 3, rồi 3 thành 2 ⇒ `final[1] = 2`. Vậy `[1, 2] → [2, 1]` ✔.

### 8.3 Trace cách 5 (giá trị gốc) trên ví dụ 2

In [14]:
origin = {v: v for v in [1, 2]}
for x, y in [[1, 3], [2, 1], [3, 2]]:
    origin[y] = origin.pop(x); print(f"{x} -> {y}: origin = {origin}  (giá trị hiện tại: giá trị ban đầu)")

1 -> 3: origin = {2: 2, 3: 1}  (giá trị hiện tại: giá trị ban đầu)
2 -> 1: origin = {3: 1, 1: 2}  (giá trị hiện tại: giá trị ban đầu)
3 -> 2: origin = {1: 2, 2: 1}  (giá trị hiện tại: giá trị ban đầu)


## 9. Code nộp LeetCode

In [15]:
from typing import List

class Solution:
    def arrayChange(self, nums: List[int], operations: List[List[int]]) -> List[int]:
        pos = {x: i for i, x in enumerate(nums)}      # giá trị -> chỉ số (giá trị phân biệt)
        for x, y in operations:
            i = pos.pop(x)                            # x chắc chắn có mặt
            nums[i] = y
            pos[y] = i                                # y chắc chắn chưa có mặt
        return nums

In [16]:
sol = Solution()
for a, ops, exp in examples:
    got = sol.arrayChange(list(a), ops)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

[3, 2, 7, 1] | kỳ vọng [3, 2, 7, 1] | OK
[2, 1] | kỳ vọng [2, 1] | OK


Bản nộp sửa trực tiếp `nums` (đề cho phép, tiết kiệm bộ nhớ); dùng `pop` để từ điển chỉ chứa đúng các giá trị đang có mặt — tiện khi muốn kiểm tra bất biến.

## 10. Vì sao thuật toán đúng

**Cách chỉ mục ngược.** Bất biến (I): `pos` là song ánh từ tập giá trị đang có trong `nums` sang tập chỉ số `{0..n−1}`, với `nums[pos[v]] = v`.

- Ban đầu đúng do giá trị phân biệt.
- Thao tác `x → y`: theo đề `x` có mặt ⇒ `i = pos[x]` là vị trí duy nhất của `x`. Gán `nums[i] = y` chính là thao tác của đề. Sau đó tập giá trị đổi từ `S` sang `S \ {x} ∪ {y}` (với `y ∉ S`), và `pos` sau `pop(x)` + `pos[y] = i` là song ánh mới, `nums[pos[y]] = y`. ⇒ (I) duy trì.

Vậy sau `m` thao tác, `nums` đúng là mảng mô phỏng.

**Cách ghép ngược.** Gọi `F_t(v)` = giá trị cuối cùng của một phần tử mang giá trị `v` ngay **trước** thao tác `t` (chỉ số từ 0). `F_m(v) = v`. Thao tác `t` là `x → y`: phần tử mang `x` thành `y` rồi chịu các thao tác `t+1…`, nên `F_t(x) = F_{t+1}(y)`; phần tử mang `v ≠ x` không đổi ở thao tác `t` nên `F_t(v) = F_{t+1}(v)`. Từ điển `final` sau khi lùi qua thao tác `t` lưu đúng `F_t` (khóa vắng mặt nghĩa là `F_t(v) = v`) — kể cả khi `x` đã có khóa từ trước (ghi đè đúng vì `F_t(x)` thay `F_{t+1}(x)`). Đáp án là `F_0(nums[i])`. ∎

Lập luận ghép ngược **không** dùng giả thiết "giá trị phân biệt" — chỉ dùng việc mỗi thao tác là một hàm trên giá trị.

**Về độ phức tạp của dict.** Mỗi thao tác của cả hai cách là O(1) phép dict **trung bình** (băm số nguyên trong Python là hàm đồng nhất, phân bố đều trên bảng); kích thước dict của chỉ mục ngược luôn đúng bằng `n` nhờ `pop`, nên bộ nhớ không phình ra dù có `10⁵` thao tác. Với cách ghép ngược, dict có tối đa `m` khóa (mỗi thao tác thêm nhiều nhất một khóa).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. `list.index` | O(n·m) | O(n) |
| 1b. Hàm hợp từng phần tử | O(n·m) | O(n) |
| 2. Chỉ mục ngược (dict) | O(n + m) | O(n) |
| 3. Ghép ngược | O(n + m) | O(m) |
| 4. Chỉ mục ngược (mảng 10⁶) | O(n + m + V) | O(V) |
| 5. Giá trị gốc | O(n + m) | O(n) |

In [17]:
def gen_case_fast(rng, n, m, MAXV=10**6):
    vals = rng.sample(range(1, MAXV + 1), n); cur = list(vals); present = set(vals); ops = []
    for _ in range(m):
        i = rng.randrange(n); x = cur[i]
        y = rng.randint(1, MAXV)
        while y in present: y = rng.randint(1, MAXV)
        ops.append([x, y]); present.discard(x); present.add(y); cur[i] = y
    return vals, ops, cur

rng = random.Random(2295)
vals, ops, expected = gen_case_fast(rng, 10**5, 10**5)
for name, f in [("index_map", index_map), ("reverse_compose", reverse_compose), ("index_array", index_array),
                ("forward_origin", forward_origin), ("Solution", lambda a, o: Solution().arrayChange(list(a), o))]:
    t0 = time.perf_counter(); r = f(vals, ops); dt = time.perf_counter() - t0
    print(f"{name:<16} {dt*1000:7.1f} ms  đúng: {r == expected}")
t0 = time.perf_counter(); brute(vals[:3000], [[x, y] for x, y in zip(vals[:3000], range(2 * 10**6, 2 * 10**6 + 3000))])
print(f"brute (n = m = 3000): {(time.perf_counter() - t0)*1000:.0f} ms")

index_map           74.9 ms  đúng: True
reverse_compose     40.9 ms  đúng: True
index_array         35.0 ms  đúng: True
forward_origin      82.3 ms  đúng: True


Solution            67.9 ms  đúng: True
brute (n = m = 3000): 27 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
ALL = {"brute": brute, "brute_compose": brute_compose, "index_map": index_map, "reverse_compose": reverse_compose,
       "index_array": index_array, "forward_origin": forward_origin,
       "Solution": lambda a, o: Solution().arrayChange(list(a), o)}
edge = examples + [
    ([5], [[5, 6]], [6]),
    ([5], [[5, 6], [6, 5]], [5]),                          # quay về giá trị cũ
    ([5], [[5, 6], [6, 7], [7, 8]], [8]),                  # chuỗi trên cùng một vị trí
    ([1, 2, 3], [[1, 4], [2, 1], [3, 2], [4, 3]], [3, 1, 2]),   # xoay vòng giá trị
    ([1, 2], [[2, 3], [1, 2], [3, 1]], [2, 1]),            # hoán đổi qua biến tạm
    ([10**6], [[10**6, 1]], [1]),
]
for a, ops, exp in edge:
    for name, f in ALL.items():
        got = f(a, ops)
        assert got == exp, (name, a, ops, got, exp)
    print(f"{str(a):<10} {str(ops):<40} -> {exp}")
print("edge OK")

[1, 2, 4, 6] [[1, 3], [4, 7], [6, 1]]                 -> [3, 2, 7, 1]
[1, 2]     [[1, 3], [2, 1], [3, 2]]                 -> [2, 1]
[5]        [[5, 6]]                                 -> [6]
[5]        [[5, 6], [6, 5]]                         -> [5]
[5]        [[5, 6], [6, 7], [7, 8]]                 -> [8]
[1, 2, 3]  [[1, 4], [2, 1], [3, 2], [4, 3]]         -> [3, 1, 2]
[1, 2]     [[2, 3], [1, 2], [3, 1]]                 -> [2, 1]
[1000000]  [[1000000, 1]]                           -> [1]
edge OK


### 12.2 Đối chiếu ngẫu nhiên với mô phỏng (seed cố định)

Giá trị lấy từ miền nhỏ `[1, 12]` để thường xuyên xảy ra "giá trị bị thay đi rồi xuất hiện lại".

In [19]:
rng = random.Random(42)
for _ in range(3000):
    n = rng.randint(1, 6); vals, ops, expected = gen_case_fast(rng, n, rng.randint(1, 10), MAXV=12)
    for name, f in ALL.items():
        assert f(vals, ops) == expected, (name, vals, ops)
print("3000 bộ ngẫu nhiên OK")

3000 bộ ngẫu nhiên OK


### 12.3 Tính chất: kết quả vẫn phân biệt; `pos` sau cùng khớp mảng; thêm thao tác "đi rồi về" không đổi kết quả

In [20]:
rng = random.Random(7)
for _ in range(1000):
    n = rng.randint(1, 8); vals, ops, expected = gen_case_fast(rng, n, rng.randint(1, 12), MAXV=20)
    out = Solution().arrayChange(list(vals), ops)
    assert len(set(out)) == len(out)
    pos = {x: i for i, x in enumerate(vals)}
    for x, y in ops: pos[y] = pos.pop(x)
    assert all(out[i] == v for v, i in pos.items())
    present = set(out); x = out[0]
    z = next(v for v in range(1, 100) if v not in present)
    assert Solution().arrayChange(list(vals), ops + [[x, z], [z, x]]) == out
print("tính chất OK")

tính chất OK


### 12.4 Edge cực trị: một vị trí bị thay 10⁵ lần, giá trị quay vòng giữa hai số

In [21]:
ops = [[1, 2] if t % 2 == 0 else [2, 1] for t in range(10**5)]       # 1->2->1->2... (chẵn số lần => về 1)
nums = [1, 3, 4]
for name, f in ALL.items():
    if name.startswith("brute"): continue
    assert f(nums, ops) == [1, 3, 4], name
ops2 = ops + [[1, 5]]
assert Solution().arrayChange(list(nums), ops2) == reverse_compose(nums, ops2) == [5, 3, 4]
big = list(range(1, 10**5 + 1)); rot = [[v, v + 10**5] for v in big]       # mọi phần tử bị thay đúng một lần
assert Solution().arrayChange(list(big), rot) == [v + 10**5 for v in big]
print("cực trị OK: chuỗi quay vòng 10^5 bước và 10^5 thao tác trên 10^5 vị trí")

cực trị OK: chuỗi quay vòng 10^5 bước và 10^5 thao tác trên 10^5 vị trí


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dựng một từ điển `x → y` rồi chỉ tra **một bước** cho mỗi phần tử ban đầu

In [22]:
def wrong_one_step(nums, operations):
    mp = {x: y for x, y in operations}
    return [mp.get(v, v) for v in nums]                    # SAI: bỏ qua các bước tiếp theo trong chuỗi

for a, ops, exp in examples:
    print("bản sai :", wrong_one_step(a, ops), "| bản đúng:", Solution().arrayChange(list(a), ops))

bản sai : [3, 2, 7, 1] | bản đúng: [3, 2, 7, 1]
bản sai : [3, 1] | bản đúng: [2, 1]


**Vì sao sai:** ví dụ 1 **trùng hợp** đúng vì mỗi phần tử **ban đầu** chỉ bị thay đúng một lần (`1 → 3`, `4 → 7`, `6 → 1`); giá trị 1 được tạo lại ở cuối nhưng không bị thay tiếp. Ví dụ 2: phần tử ban đầu 1 thành 3 rồi thành 2, nhưng tra một bước chỉ ra 3; phần tử 2 tra một bước ra 1 — đúng một cách tình cờ. **Sửa:** dùng chỉ mục ngược hoặc ghép ngược đủ mọi bước.

### Lỗi 2 — Đi theo chuỗi `x → y → …` trong từ điển tĩnh (gặp **chu trình** hoặc đi quá)

In [23]:
def wrong_follow_chain(nums, operations, limit=50):
    mp = {}
    for x, y in operations: mp[x] = y                      # thao tác sau ghi đè thao tác trước cùng khóa
    out = []
    for v in nums:
        steps = 0
        while v in mp and steps < limit:                   # SAI: không biết bước nào xảy ra SAU bước nào
            v = mp[v]; steps += 1
        out.append(v if steps < limit else "VÒNG LẶP")
    return out

for a, ops, exp in examples:
    print("bản sai :", wrong_follow_chain(a, ops), "| bản đúng:", Solution().arrayChange(list(a), ops))

bản sai : [3, 2, 7, 3] | bản đúng: [3, 2, 7, 1]
bản sai : ['VÒNG LẶP', 'VÒNG LẶP'] | bản đúng: [2, 1]


**Vì sao sai:** ví dụ 2 có `1 → 3`, `2 → 1`, `3 → 2` — từ điển tĩnh tạo chu trình `1 → 3 → 2 → 1 → …`, vòng `while` không dừng (đã giới hạn 50 bước để minh họa). Từ điển tĩnh mất **thứ tự thời gian**: bước `2 → 1` xảy ra **trước** `3 → 2` nhưng sau `1 → 3`. Ví dụ 1 còn đi quá: 1 → 3 đúng, nhưng 6 → 1 → 3 (sai, phải dừng ở 1). **Sửa:** chỉ mục ngược, hoặc ghép ngược theo thời gian.

### Lỗi 3 — Cập nhật `nums` nhưng quên cập nhật `pos[y]`

In [24]:
def wrong_no_update(nums, operations):
    nums = list(nums); pos = {x: i for i, x in enumerate(nums)}
    try:
        for x, y in operations:
            nums[pos[x]] = y                               # SAI: không ghi pos[y]
        return nums
    except KeyError as e:
        return f"KeyError {e}"

for a, ops, exp in examples:
    print("bản sai :", wrong_no_update(a, ops), "| bản đúng:", Solution().arrayChange(list(a), ops))

bản sai : [3, 2, 7, 1] | bản đúng: [3, 2, 7, 1]
bản sai : KeyError 3 | bản đúng: [2, 1]


**Vì sao sai:** ví dụ 1 **trùng hợp** đúng (không thao tác nào thay một giá trị vừa được tạo ra… `6 → 1`: `pos[6]` vẫn đúng). Ví dụ 2: thao tác `3 → 2` cần `pos[3]`, nhưng 3 được tạo ra ở thao tác đầu và chưa từng được ghi vào `pos` ⇒ `KeyError`. **Sửa:** `pos[y] = pos[x]`.

### Lỗi 4 — Ghép ngược nhưng duyệt **xuôi**

In [25]:
def wrong_forward_compose(nums, operations):
    final = {}
    for x, y in operations:                                # SAI: phải duyệt ngược
        final[x] = final.get(y, y)
    return [final.get(v, v) for v in nums]

for a, ops, exp in examples:
    print("bản sai :", wrong_forward_compose(a, ops), "| bản đúng:", Solution().arrayChange(list(a), ops))

bản sai : [3, 2, 7, 3] | bản đúng: [3, 2, 7, 1]
bản sai : [3, 3] | bản đúng: [2, 1]


**Vì sao sai:** `final.get(y, y)` phải phản ánh những gì xảy ra **sau** thao tác hiện tại; duyệt xuôi thì `final[y]` (nếu có) mô tả những gì xảy ra **trước** đó. Ví dụ 1: duyệt xuôi đặt `final[6] = final.get(1) = 3` (vì `1 → 3` xảy ra trước), nên 6 bị biến thành 3 thay vì 1. Ví dụ 2 cho `[3, 3]` — mảng kết quả còn có hai giá trị trùng nhau, điều không thể xảy ra theo đề.

### Lỗi 5 — Dùng `del pos[x]` **trước** khi đọc chỉ số

In [26]:
def wrong_del_first(nums, operations):
    nums = list(nums); pos = {x: i for i, x in enumerate(nums)}
    try:
        for x, y in operations:
            del pos[x]
            nums[pos[x]] = y; pos[y] = pos[x]               # SAI: pos[x] đã bị xóa
        return nums
    except KeyError as e:
        return f"KeyError {e}"

print("bản sai :", wrong_del_first(*examples[0][:2]), "| bản đúng:", Solution().arrayChange(list(examples[0][0]), examples[0][1]))

bản sai : KeyError 1 | bản đúng: [3, 2, 7, 1]


**Sửa:** `i = pos.pop(x)` — lấy và xóa trong một bước.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tìm vị trí của một giá trị" lặp lại nhiều lần, giá trị **phân biệt** ⇒ **từ điển giá trị → chỉ số**, nhớ cập nhật song song.
- Chuỗi thay thế giá trị theo thời gian ⇒ **ghép ánh xạ ngược** (từ cuối về đầu) — không cần mô phỏng, không bị chu trình.
- Từ điển tĩnh `x → y` **mất thứ tự thời gian** ⇒ không đi theo chuỗi được.
- Giá trị ≤ 10⁶ ⇒ mảng thay dict nếu cần tốc độ.
- Bài cùng họ: 1. Two Sum (chỉ mục ngược), 380 Insert Delete GetRandom O(1) (mảng + dict song song), 1409 Queries on a Permutation With Key, 2671 Frequency Tracker, 990 Satisfiability of Equality Equations.

## 15. Biến thể của bài

### 15.1 Biến thể A — Truy vấn trực tuyến xen kẽ: "giá trị `v` đang ở chỉ số nào?" / "chỉ số `i` đang mang giá trị gì?"

Chỉ mục ngược trả lời cả hai trong O(1); cách ghép ngược thì không (cần biết toàn bộ thao tác trước).

In [27]:
class Tracker:
    def __init__(self, nums):
        self.nums = list(nums); self.pos = {x: i for i, x in enumerate(nums)}
    def replace(self, x, y):
        i = self.pos.pop(x); self.nums[i] = y; self.pos[y] = i
    def where(self, v):
        return self.pos.get(v, -1)
    def at(self, i):
        return self.nums[i]

t = Tracker([1, 2, 4, 6]); t.replace(1, 3); print(t.where(3), t.where(1), t.at(0)); t.replace(6, 1); print(t.where(1), t.nums)

0 -1 3
3 [3, 2, 4, 1]


### 15.2 Biến thể B — Mảng có **giá trị trùng**, thao tác thay **mọi** lần xuất hiện của `x`, `y` có thể đã tồn tại

Chỉ mục ngược phải gộp danh sách chỉ số (gộp nhỏ vào lớn để đạt O((n + m) log n)); cách ghép ngược **vẫn đúng nguyên vẹn** vì nó chỉ coi thao tác là hàm trên giá trị.

In [28]:
def replace_all_brute(nums, operations):
    a = list(nums)
    for x, y in operations:
        a = [y if v == x else v for v in a]
    return a

def replace_all_small_to_large(nums, operations):
    where = {}
    for i, v in enumerate(nums): where.setdefault(v, []).append(i)
    for x, y in operations:
        if x == y or x not in where: continue
        src = where.pop(x); dst = where.get(y)
        if dst is None: where[y] = src
        else:
            if len(src) > len(dst): src, dst = dst, src
            dst.extend(src); where[y] = dst
    out = [0] * len(nums)
    for v, idx in where.items():
        for i in idx: out[i] = v
    return out

rng = random.Random(152)
for _ in range(3000):
    a = [rng.randint(1, 5) for _ in range(rng.randint(1, 8))]
    ops = [[rng.randint(1, 6), rng.randint(1, 6)] for _ in range(rng.randint(1, 8))]
    ref = replace_all_brute(a, ops)
    assert reverse_compose(a, ops) == ref == replace_all_small_to_large(a, ops)
print(reverse_compose([1, 1, 2, 3], [[1, 2], [2, 3], [3, 1]]), replace_all_brute([1, 1, 2, 3], [[1, 2], [2, 3], [3, 1]]))

[1, 1, 1, 1] [1, 1, 1, 1]


### 15.3 Biến thể C — Thao tác **hoán đổi** hai giá trị `x ↔ y` (cả hai đều có mặt)

In [29]:
def apply_swaps(nums, swaps):
    nums = list(nums); pos = {x: i for i, x in enumerate(nums)}
    for x, y in swaps:
        i, j = pos[x], pos[y]
        nums[i], nums[j] = y, x; pos[x], pos[y] = j, i
    return nums

def apply_swaps_brute(nums, swaps):
    a = list(nums)
    for x, y in swaps:
        i, j = a.index(x), a.index(y); a[i], a[j] = a[j], a[i]
    return a

rng = random.Random(153)
for _ in range(1000):
    a = rng.sample(range(1, 10), rng.randint(2, 6))
    sw = [rng.sample(a, 2) for _ in range(rng.randint(1, 6))]
    assert apply_swaps(a, sw) == apply_swaps_brute(a, sw)
print(apply_swaps([1, 2, 3, 4], [[1, 4], [2, 4]]))

[2, 4, 3, 1]


### 15.4 Biến thể D — Hoàn tác (undo) thao tác cuối

Lưu ngăn xếp các thao tác đã làm; hoàn tác `x → y` chính là thực hiện `y → x` (hợp lệ vì sau thao tác, `y` có mặt và `x` không có).

In [30]:
class UndoTracker(Tracker):
    def __init__(self, nums):
        super().__init__(nums); self.hist = []
    def replace(self, x, y):
        super().replace(x, y); self.hist.append((x, y))
    def undo(self):
        x, y = self.hist.pop(); Tracker.replace(self, y, x)

u = UndoTracker([1, 2]); u.replace(1, 3); u.replace(2, 1); print(u.nums); u.undo(); print(u.nums); u.undo(); print(u.nums)

[3, 1]
[3, 2]
[1, 2]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `list.index` | định nghĩa | O(n·m) | O(n) | kiểm chứng |
| 2. Chỉ mục ngược | dict giá trị → chỉ số | O(n+m) | O(n) | **mặc định**, hỗ trợ truy vấn trực tuyến |
| 3. Ghép ngược | hàm hợp từ cuối | O(n+m) | O(m) | giá trị có thể trùng / offline |
| 4. Mảng 10⁶ | như 2 nhưng mảng | O(n+m+V) | O(V) | cần tốc độ thô |
| 5. Giá trị gốc | theo dõi nguồn gốc | O(n+m) | O(n) | minh họa ánh xạ ngược |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `arrayChange([7, 8, 9], [[8, 1], [7, 8], [1, 7]])`?

**Bài 2 (tính tay):** Trace cách ghép ngược cho `nums = [4, 5]`, `operations = [[4, 6], [6, 4], [5, 6]]`.

**Bài 3 (điền TODO):** Viết `bt3_todo(nums, operations)` trả về danh sách **chỉ số bị thay** ở từng thao tác (độ dài `m`).

**Bài 4 (tự viết lại):** Viết `bt4_todo(nums, operations)` cài cách ghép ngược không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(nums, operations)` trả về số **giá trị khác nhau** từng xuất hiện ở chỉ số 0 trong suốt quá trình (kể cả giá trị ban đầu).

In [31]:
def bt3_todo(nums, operations):
    # TODO: chỉ số bị thay ở từng thao tác
    pass

def bt4_todo(nums, operations):
    # TODO: ghép ngược
    pass

def bt5_todo(nums, operations):
    # TODO: số giá trị khác nhau từng ở chỉ số 0
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `8 → 1`: `[7, 1, 9]`. Mức 2: `7 → 8`: `[8, 1, 9]`. Mức 3: `1 → 7`: `[8, 7, 9]`.

**Bài 2.** Mức 1: lùi qua `5 → 6`: `final[5] = 6`. Mức 2: lùi qua `6 → 4`: `final[6] = 4`. Mức 3: lùi qua `4 → 6`: `final[4] = final[6] = 4` ⇒ `[4, 6]`.

**Bài 3.** Mức 1: dùng `pos`. Mức 2: ghi `pos[x]` trước khi cập nhật. Mức 3: `out.append(i)`.

**Bài 4.** Mức 1: `reversed(operations)`. Mức 2: `final[x] = final.get(y, y)`. Mức 3: `[final.get(v, v) for v in nums]`.

**Bài 5.** Mức 1: chỉ quan tâm các thao tác có `pos[x] == 0`. Mức 2: lưu tập giá trị. Mức 3: bắt đầu bằng `{nums[0]}`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `[8, 7, 9]`. **Bài 2:** `[4, 6]`.

In [32]:
assert Solution().arrayChange([7, 8, 9], [[8, 1], [7, 8], [1, 7]]) == [8, 7, 9]
assert reverse_compose([4, 5], [[4, 6], [6, 4], [5, 6]]) == [4, 6]

def bt3(nums, operations):
    pos = {x: i for i, x in enumerate(nums)}; out = []
    for x, y in operations:
        i = pos.pop(x); out.append(i); pos[y] = i
    return out

def bt4(nums, operations):
    final = {}
    for x, y in reversed(operations): final[x] = final.get(y, y)
    return [final.get(v, v) for v in nums]

def bt5(nums, operations):
    pos = {x: i for i, x in enumerate(nums)}; seen = {nums[0]}
    for x, y in operations:
        i = pos.pop(x); pos[y] = i
        if i == 0: seen.add(y)
    return len(seen)

rng = random.Random(1990)
for _ in range(2000):
    n = rng.randint(1, 6); vals, ops, expected = gen_case_fast(rng, n, rng.randint(1, 10), MAXV=12)
    a = list(vals); idxs = []; seen0 = {a[0]}
    for x, y in ops:
        i = a.index(x); idxs.append(i); a[i] = y
        if i == 0: seen0.add(y)
    assert bt3(vals, ops) == idxs and bt4(vals, ops) == expected and bt5(vals, ops) == len(seen0)
print(bt3(*examples[0][:2]), bt5(*examples[1][:2]))
print("Đáp án bài tập: tất cả assert OK")

[0, 2, 3] 3
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
pos = {x: i for i, x in enumerate(nums)}   # giá trị phân biệt => song ánh
for x, y in operations:
    i = pos.pop(x)                         # vị trí của x
    nums[i] = y
    pos[y] = i                             # giữ bất biến
return nums

# offline / giá trị có thể trùng:
final = {}
for x, y in reversed(operations): final[x] = final.get(y, y)
return [final.get(v, v) for v in nums]
```

**Checklist:**
- [ ] Không dùng `list.index` trong vòng lặp (O(n·m)).
- [ ] Cập nhật **cả** `nums[i]` và `pos[y]`.
- [ ] Không đi theo chuỗi trong từ điển tĩnh `x → y` (mất thứ tự thời gian, có thể lặp vô hạn).
- [ ] Ghép ánh xạ thì duyệt **ngược**.